# Applying Models for Code Generation (OpenAI Codex, GitHub Copilot)

## 📚 Learning Objectives

By completing this notebook, you will:
- Explain how Codex/Copilot-class models generate code (autoregressive LMs trained on source code)
- Train a tiny **character-level model on Python source** and generate code with it
- **Evaluate** the generated code the way code models are evaluated: does it parse? does it run?
- Understand what separates the toy from the real thing (scale, tokenizers, execution-based
  benchmarks like HumanEval)

## 🔗 Where this fits

**Builds on:** Course 10 — Unit 2, lessons 01 and 03 (autoregressive language models, and prompting) — Copilot is that model trained on source code and prompted with your editor buffer.

**Used later in:** Course 12 (AIAT 126) — Unit 3, where these tools are used — and their output verified — during implementation.

---

This notebook covers practical activities from **Course 10, Unit 5**:
- Applying models like OpenAI Codex or GitHub Copilot for code generation

---

## Introduction

**Code generation models** like Codex and Copilot are ordinary autoregressive language models
trained on billions of lines of source code — to them, Python is just text with unusually strict
grammar. We exploit that here: the same char-level LM you used for prose, retrained on Python.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# WHAT/WHY: orientation cell — what Codex and Copilot do, before the
# hands-on next-token generation demo below.
print("✅ Libraries imported!")
print("\nCode Generation Models")
print("=" * 60)

print("\nOpenAI Codex:")
print("  - Code generation")
print("  - Multiple languages")
print("  - Natural language input")
print("  - API access")

print("\nGitHub Copilot:")
print("  - IDE integration")
print("  - Context-aware")
print("  - Real-time suggestions")
print("  - Code completion")

print("\nApplications:")
print("  - Code completion")
print("  - Function generation")
print("  - Bug fixing")
print("  - Documentation")

print("\n✅ Code generation concepts understood!")

✅ Libraries imported!

Code Generation Models

OpenAI Codex:
  - Code generation
  - Multiple languages
  - Natural language input
  - API access

GitHub Copilot:
  - IDE integration
  - Context-aware
  - Real-time suggestions
  - Code completion

Applications:
  - Code completion
  - Function generation
  - Bug fixing
  - Documentation

✅ Code generation concepts understood!


## 🌍 Worked Example — a Character-Level Model Trained on Python

**Industry context:**
- GitHub Copilot autocompletes code token by token from your file's context
- Codex (the model behind early Copilot) was GPT-3 fine-tuned on public code
- Modern coding models are evaluated by *executing* their output (HumanEval, MBPP benchmarks)

We train the course's character-level LM on a tiny corpus of **Python functions** and then ask it
to complete `def add(a, b):` — real code generation, at one-millionth the scale. Afterwards we
evaluate the output the honest way: by checking whether Python can actually parse it.

In [2]:
# Train a tiny character-level LM on Python source code, generate a completion, and
# evaluate it with a parse check. WHY: this is Copilot's recipe in miniature — and the
# evaluation shows exactly why code models need scale.
import torch, torch.nn as nn, torch.optim as optim
import numpy as np

torch.manual_seed(42)
# ── Training corpus: real (tiny) Python functions, repeated for more windows ─
snippet = (
    "def add(a, b):\n    return a + b\n\n"
    "def sub(a, b):\n    return a - b\n\n"
    "def mul(a, b):\n    return a * b\n\n"
    "def square(a):\n    return a * a\n\n"
    "def half(a):\n    return a / 2\n\n"
)
text = snippet * 3

# ── Tokenize: every distinct character is a token (newlines and colons too) ─
chars  = sorted(set(text))
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for c, i in c2i.items()}
VOCAB  = len(chars)
enc    = [c2i[c] for c in text]
print(f"Vocabulary: {VOCAB} characters (letters, digits, newline, ':', '+', ...)")

# ── Training windows: 20 characters in -> next character out ──────────────
SEQ_LEN = 20
X_list, y_list = [], []
for i in range(len(enc) - SEQ_LEN - 1):
    X_list.append(enc[i:i + SEQ_LEN])
    y_list.append(enc[i + SEQ_LEN])
X_t = torch.tensor(X_list, dtype=torch.long)
y_t = torch.tensor(y_list, dtype=torch.long)

# ── Same LSTM language model as Unit 2 — only the corpus changed ──────────
class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, 32)
        self.lstm  = nn.LSTM(32, 128, batch_first=True, num_layers=2)
        self.fc    = nn.Linear(128, VOCAB)
    def forward(self, x):
        out, _ = self.lstm(self.embed(x))
        return self.fc(out[:, -1, :])              # logits for the next character

model   = CharLM()
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

# ── Train with mini-batches for 200 steps ─────────────────────────────────
for epoch in range(200):
    model.train()
    perm = torch.randperm(len(X_t))[:256]
    loss = loss_fn(model(X_t[perm]), y_t[perm])
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f"Epoch {epoch} — loss: {loss.item():.3f}")

# ── Generate: complete a function header, sampling with temperature ───────
def generate(seed_str, steps=80, temperature=0.5):
    model.eval()
    chars_out = list(seed_str)
    ctx = [c2i.get(c, 0) for c in seed_str[-SEQ_LEN:]]
    for _ in range(steps):
        inp = torch.tensor([ctx[-SEQ_LEN:]]).long()
        with torch.no_grad():
            logits = model(inp)[0] / temperature
        probs = torch.softmax(logits, 0).numpy()
        next_c = np.random.choice(len(probs), p=probs)
        chars_out.append(i2c[next_c])
        ctx.append(next_c)
    return ''.join(chars_out)

np.random.seed(0)
completion = generate("def add(a, b):\n", steps=60, temperature=0.5)
print("\n── Generated Python ────────────────────────────────────────────")
print(completion)

# ── Evaluate like a code model: can Python parse it? (computed, not assumed) ─
# We cut at the last blank line so we judge only complete function blocks.
candidate = completion.rsplit("\n\n", 1)[0]
import ast
try:
    ast.parse(candidate)
    print("\n✅ ast.parse: the generated snippet IS syntactically valid Python.")
except SyntaxError as e:
    print(f"\n❌ ast.parse: SyntaxError ({e.msg}, line {e.lineno}) — the model produced")
    print("   code-LOOKING text that Python rejects.")

# ── Evaluate SEMANTICS: execute it and run a unit test (HumanEval in miniature) ─
# Syntax can pass while the logic is wrong — only a test can tell the difference.
ns = {}
try:
    exec(candidate, ns)                                  # run the generated code
    if 'add' in ns:
        result = ns['add'](2, 3)
        verdict = "✅ PASSED" if result == 5 else "❌ FAILED"
        print(f"\nUnit test add(2, 3) == 5 → got {result} → {verdict}")
    else:
        print("\nUnit test skipped: the snippet did not define add().")
except Exception as e:
    print(f"\nExecution error: {type(e).__name__}: {e}")

print("\nThe lesson: correctness must be CHECKED, never assumed — real code models are")
print("graded by executing outputs against tests (HumanEval), and even Copilot-class")
print("systems make errors, which is why their suggestions always need review.")

Vocabulary: 25 characters (letters, digits, newline, ':', '+', ...)


Epoch 0 — loss: 3.234


Epoch 50 — loss: 0.471


Epoch 100 — loss: 0.069


Epoch 150 — loss: 0.035



── Generated Python ────────────────────────────────────────────
def add(a, b):
    return a - b

def mul(a, b):
    return a * b

def squar

✅ ast.parse: the generated snippet IS syntactically valid Python.

Unit test add(2, 3) == 5 → got -1 → ❌ FAILED

The lesson: correctness must be CHECKED, never assumed — real code models are
graded by executing outputs against tests (HumanEval), and even Copilot-class
systems make errors, which is why their suggestions always need review.


## 📚 References & Further Reading

**Papers:**
- Chen et al. (2021) — [Evaluating Large Language Models Trained on Code](https://arxiv.org/abs/2107.03374) *(the Codex paper; introduced HumanEval)*
- Radford et al. (2019) — [GPT-2](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

**Interactive:**
- [Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT) — build GPT in 300 lines
- [GitHub Copilot documentation](https://docs.github.com/en/copilot)

**State-of-the-Art:** modern coding assistants pair code-trained LLMs with retrieval over your
repository and execution feedback; benchmarks (HumanEval, SWE-bench) grade them by running tests.

## 📝 Summary

In this notebook, you:

- Saw that Codex/Copilot-class systems are **autoregressive language models trained on source
  code** — the same mechanism as Unit 2's text models
- **Trained a char-level LM on Python functions** and generated a completion for `def add(a, b):`
- **Evaluated** the output with a parse check (`ast.parse`) — the small-scale version of the
  execution-based benchmarks (HumanEval) used on real code models
- Learned what separates the toy from production: massive code corpora, subword tokenizers,
  repository context, and test-based evaluation

**Key takeaway:** generated code is a *proposal* — always validated by parsing, tests, and review.